# Blok D — Rancangan reward: PURE3 vs ASLI

**PURE3** (3 aliran): `wait` · `gini` · `acceptance`, satu suku per aliran. Bobot `alpha`
lenyap secara struktural — penskala seragam satu-suku hilang baik di normalisasi
advantage maupun di gap-ratio.

**ASLI** (2 aliran): `individual` = wait + prox (+acceptance opsional), `global` = gini +
flock (+acceptance opsional). Menuntut kalibrasi `alpha` antar-suku di dalam satu aliran.

> **Keterbatasan data.** Faktorial ASLI tidak lengkap: sel1 (`_noattn`) dan sel2
> (`(baku)`) tidak pernah dievaluasi pada varian tanpa-acceptance, dan sel1
> (`_noattn_acc0.5`) tidak ada pada varian ber-acceptance. Perbandingan karena itu
> dibatasi pada sel yang tersedia — **efek faktorial penuh tidak dapat dihitung untuk
> ASLI**, hanya perbandingan sel-per-sel.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import _analisis_bab5 as A
pd.set_option('display.width', 230); pd.set_option('display.max_columns', 50)
plt.rcParams.update({'figure.dpi': 110, 'font.size': 9, 'axes.grid': True, 'grid.alpha': .3})
HORIZON = '90d'
B = 'master_hybrid_ppo_dgr_90d_cwtfail120pen-2_preffeat_pairout'
A.ARAH_BAIK.update({'gini_UTIL': 'min', 'gini_ANTRE': 'min'})

def nilai(tag, m, lengan=None):
    if m == 'gini_UTIL':  return A.gini_stasiun(tag, 'gini_utilisasi', HORIZON, lengan)
    if m == 'gini_ANTRE': return A.gini_stasiun(tag, 'gini_antrean',   HORIZON, lengan)
    return A.unit_stat(tag, m, HORIZON, lengan)

PURE3 = {'sel1': f'{B}_noattn_pure3', 'sel2': f'{B}_pure3',
         'sel3': f'{B}_pg0.1_noattn_pure3', 'sel4': f'{B}_pg0.1_pure3'}
ASLI  = {'sel3': f'{B}_pg0.1_noattn', 'sel4': f'{B}_pg0.1'}          # tanpa acceptance
ASLIA = {'sel2': f'{B}_acc0.5', 'sel3': f'{B}_pg0.1_noattn_acc0.5',
         'sel4': f'{B}_pg0.1_acc0.5'}                                  # + acceptance
for nm, d in [('PURE3', PURE3), ('ASLI', ASLI), ('ASLI+acc', ASLIA)]:
    print(f'{nm:9s} sel tersedia: {sorted(d)}')

PURE3     sel tersedia: ['sel1', 'sel2', 'sel3', 'sel4']
ASLI      sel tersedia: ['sel3', 'sel4']
ASLI+acc  sel tersedia: ['sel2', 'sel3', 'sel4']


## D.1 — Perbandingan sel yang tersedia di kedua rancangan

Hanya `sel3` dan `sel4` ada di ketiga varian, jadi itu basis perbandingan yang sah.

In [2]:
M = ['gini_UTIL', 'gini_ANTRE', 'wait', 'acc', 'trust', 'rec_entropy', 'herding',
     'flocking', 'jnpatuh_expost_frac_untung', 'jnpatuh_expost_mean']
baris = []
for rancangan, d in [('PURE3', PURE3), ('ASLI', ASLI), ('ASLI+acc', ASLIA)]:
    for sel, t in sorted(d.items()):
        r = {'rancangan': rancangan, 'sel': sel}
        for m in M:
            r[m] = nilai(t, m)[0]
        baris.append(r)
tab = pd.DataFrame(baris).set_index(['rancangan', 'sel'])
display(tab.round(4))

gini_UTIL  gini_ANTRE     wait     acc   trust  rec_entropy  herding  flocking  jnpatuh_expost_frac_untung  jnpatuh_expost_mean
rancangan sel                                                                                                                                  
PURE3     sel1     0.0289      0.0929  95.9557  0.6700  0.3666       0.0602   0.3658    0.1158                      0.5298              17.6033
          sel2     0.0393      0.1216  89.6613  0.7021  0.3935       0.0788   0.3471    0.0909                      0.4995              17.3210
          sel3     0.0521      0.1447  84.1083  0.7034  0.3960       0.0747   0.3499    0.0916                      0.5113              17.6827
          sel4     0.0206      0.0690  66.2480  0.7194  0.4348       0.1182   0.3063    0.0687                      0.5385              20.6793
ASLI      sel3     0.0302      0.0696  72.8817  0.7034  0.4171       0.1169   0.3014    0.0953                      0.5704              25.5179
          sel4     0.0241      0.0590  69.0531  0.7152  0.4292       0.1355   0.2814    0.0828                      0.5623              24.9837
ASLI+acc  sel2     0.0282      0.0594  68.2709  0.7149  0.4330       0.1549   0.2592    0.0839                      0.5516              26.0830
          sel3     0.0242      0.0610  67.7453  0.7181  0.4342       0.1374   0.2797    0.0802                      0.5387              25.3375
          sel4     0.0250      0.0600  67.7464  0.7159  0.4343       0.1195   0.3010    0.0737                      0.5673              25.4605

In [3]:
print('SEL YANG SAMA, RANCANGAN BERBEDA — selisih PURE3 relatif thd ASLI\n')
for sel in ['sel3', 'sel4']:
    print(f'--- {sel} ---')
    for m in M:
        p = nilai(PURE3[sel], m)[0]
        a = nilai(ASLI[sel], m)[0]
        aa = nilai(ASLIA[sel], m)[0]
        arah = A.ARAH_BAIK.get(m, 'max')
        lbh = lambda x, y: 'PURE3' if ((x < y) == (arah == 'min')) else 'ASLI'
        print(f'   {m:28s} PURE3={p:9.4f}  ASLI={a:9.4f} ({lbh(p,a)})  '
              f'ASLI+acc={aa:9.4f} ({lbh(p,aa)})')
    print()

SEL YANG SAMA, RANCANGAN BERBEDA — selisih PURE3 relatif thd ASLI

--- sel3 ---
   gini_UTIL                    PURE3=   0.0521  ASLI=   0.0302 (ASLI)  ASLI+acc=   0.0242 (ASLI)
   gini_ANTRE                   PURE3=   0.1447  ASLI=   0.0696 (ASLI)  ASLI+acc=   0.0610 (ASLI)
   wait                         PURE3=  84.1083  ASLI=  72.8817 (ASLI)  ASLI+acc=  67.7453 (ASLI)
   acc                          PURE3=   0.7034  ASLI=   0.7034 (ASLI)  ASLI+acc=   0.7181 (ASLI)
   trust                        PURE3=   0.3960  ASLI=   0.4171 (ASLI)  ASLI+acc=   0.4342 (ASLI)
   rec_entropy                  PURE3=   0.0747  ASLI=   0.1169 (ASLI)  ASLI+acc=   0.1374 (ASLI)
   herding                      PURE3=   0.3499  ASLI=   0.3014 (ASLI)  ASLI+acc=   0.2797 (ASLI)
   flocking                     PURE3=   0.0916  ASLI=   0.0953 (PURE3)  ASLI+acc=   0.0802 (ASLI)
   jnpatuh_expost_frac_untung   PURE3=   0.5113  ASLI=   0.5704 (ASLI)  ASLI+acc=   0.5387 (ASLI)


   jnpatuh_expost_mean          PURE3=  17.6827  ASLI=  25.5179 (ASLI)  ASLI+acc=  25.3375 (ASLI)

--- sel4 ---
   gini_UTIL                    PURE3=   0.0206  ASLI=   0.0241 (PURE3)  ASLI+acc=   0.0250 (PURE3)
   gini_ANTRE                   PURE3=   0.0690  ASLI=   0.0590 (ASLI)  ASLI+acc=   0.0600 (ASLI)
   wait                         PURE3=  66.2480  ASLI=  69.0531 (PURE3)  ASLI+acc=  67.7464 (PURE3)
   acc                          PURE3=   0.7194  ASLI=   0.7152 (PURE3)  ASLI+acc=   0.7159 (PURE3)
   trust                        PURE3=   0.4348  ASLI=   0.4292 (PURE3)  ASLI+acc=   0.4343 (PURE3)
   rec_entropy                  PURE3=   0.1182  ASLI=   0.1355 (ASLI)  ASLI+acc=   0.1195 (ASLI)
   herding                      PURE3=   0.3063  ASLI=   0.2814 (ASLI)  ASLI+acc=   0.3010 (ASLI)
   flocking                     PURE3=   0.0687  ASLI=   0.0828 (PURE3)  ASLI+acc=   0.0737 (PURE3)


   jnpatuh_expost_frac_untung   PURE3=   0.5385  ASLI=   0.5623 (ASLI)  ASLI+acc=   0.5673 (ASLI)
   jnpatuh_expost_mean          PURE3=  20.6793  ASLI=  24.9837 (ASLI)  ASLI+acc=  25.4605 (ASLI)



## D.2 — Ongkos yang wajib diakui: koordinasi

PURE3 membuang suku `flock` (anti-herding jendela bergulir) demi menjaga satu-suku-per-
aliran. Bila ongkosnya nyata, ia harus terlihat pada `rec_entropy` dan `herding`.

In [4]:
baris = []
for rancangan, d in [('PURE3', PURE3), ('ASLI', ASLI), ('ASLI+acc', ASLIA)]:
    for sel, t in sorted(d.items()):
        baris.append({'rancangan': rancangan, 'sel': sel,
                      'rec_entropy': nilai(t, 'rec_entropy')[0],
                      'herding': nilai(t, 'herding')[0],
                      'flocking': nilai(t, 'flocking')[0]})
ko = pd.DataFrame(baris)
display(ko.set_index(['rancangan', 'sel']).round(4))

p3 = ko[ko['rancangan'] == 'PURE3']; asli = ko[ko['rancangan'] != 'PURE3']
for m, arah in [('rec_entropy', 'max'), ('herding', 'min')]:
    lo_p, hi_p = p3[m].min(), p3[m].max()
    lo_a, hi_a = asli[m].min(), asli[m].max()
    pisah = hi_p < lo_a or hi_a < lo_p
    print(f'{m:12s} PURE3 [{lo_p:.4f}, {hi_p:.4f}]  ASLI [{lo_a:.4f}, {hi_a:.4f}]  '
          f'-> {"TERPISAH SEMPURNA" if pisah else "tumpang tindih"}')
print()
print('Bila terpisah sempurna, ongkos membuang `flock` nyata dan WAJIB dilaporkan')
print('sebagai konsekuensi rancangan, bukan disembunyikan.')

rec_entropy  herding  flocking
rancangan sel                                 
PURE3     sel1       0.0602   0.3658    0.1158
          sel2       0.0788   0.3471    0.0909
          sel3       0.0747   0.3499    0.0916
          sel4       0.1182   0.3063    0.0687
ASLI      sel3       0.1169   0.3014    0.0953
          sel4       0.1355   0.2814    0.0828
ASLI+acc  sel2       0.1549   0.2592    0.0839
          sel3       0.1374   0.2797    0.0802
          sel4       0.1195   0.3010    0.0737

rec_entropy  PURE3 [0.0602, 0.1182]  ASLI [0.1169, 0.1549]  -> tumpang tindih
herding      PURE3 [0.3063, 0.3658]  ASLI [0.2592, 0.3014]  -> TERPISAH SEMPURNA

Bila terpisah sempurna, ongkos membuang `flock` nyata dan WAJIB dilaporkan
sebagai konsekuensi rancangan, bukan disembunyikan.


## D.3 — Apakah `alpha` benar-benar lenyap di PURE3?

Klaim strukturalnya: dengan satu suku per aliran, bobot `alpha` merosot jadi penskala
seragam yang hilang di normalisasi advantage **dan** di gap-ratio. Konsekuensi yang
dapat diperiksa: bobot DGR PURE3 seharusnya **lebih stabil antar-seed** daripada ASLI,
karena tak ada proporsi antar-suku yang harus ditemukan.

In [5]:
for rancangan, d in [('PURE3', PURE3), ('ASLI', ASLI), ('ASLI+acc', ASLIA)]:
    print(f'--- {rancangan} ---')
    for sel, t in sorted(d.items()):
        try:
            b = A.beta_akhir(t)
            sd = b.iloc[:, 0].std(ddof=0)
            print(f'   {sel}: SD antar-seed bobot aliran-1 = {sd:.4f}   '
                  f'{[np.round(r.values, 3).tolist() for _, r in b.iterrows()]}')
        except Exception as e:
            print(f'   {sel}: tak dapat dibaca ({type(e).__name__})')
    print()

--- PURE3 ---
   sel1: SD antar-seed bobot aliran-1 = 0.2513   [[0.567, 0.278, 0.155], [0.048, 0.925, 0.027], [0.594, 0.194, 0.212]]
   sel2: SD antar-seed bobot aliran-1 = 0.2979   [[0.767, 0.192, 0.041], [0.044, 0.953, 0.003], [0.319, 0.619, 0.062]]
   sel3: SD antar-seed bobot aliran-1 = 0.1255   [[0.826, 0.067, 0.107], [0.62, 0.173, 0.206], [0.525, 0.464, 0.011]]
   sel4: SD antar-seed bobot aliran-1 = 0.0930   [[0.801, 0.172, 0.027], [0.6, 0.341, 0.059], [0.608, 0.336, 0.056]]

--- ASLI ---
   sel3: SD antar-seed bobot aliran-1 = 0.0912   [[0.635, 0.365], [0.831, 0.169], [0.641, 0.359]]
   sel4: SD antar-seed bobot aliran-1 = 0.0753   [[0.605, 0.395], [0.594, 0.406], [0.759, 0.241]]

--- ASLI+acc ---
   sel2: SD antar-seed bobot aliran-1 = 0.0385   [[0.538, 0.462], [0.602, 0.398], [0.63, 0.37]]
   sel3: SD antar-seed bobot aliran-1 = 0.0310   [[0.618, 0.382], [0.609, 0.391], [0.679, 0.321]]
   sel4: SD antar-seed bobot aliran-1 = 0.0563   [[0.729, 0.271], [0.732, 0.268], [0.611, 0

## D.4 — Ringkasan Blok D

In [6]:
print('RINGKASAN BLOK D'.center(74, '='))
print()
print('Basis perbandingan: sel3 & sel4 (satu-satunya yang ada di ketiga varian).')
print('Faktorial ASLI TIDAK lengkap -> efek utama & interaksi tak dapat dihitung.')
print()
for m in ['gini_UTIL', 'wait', 'acc', 'rec_entropy', 'herding']:
    p = np.mean([nilai(PURE3[s], m)[0] for s in ['sel3', 'sel4']])
    a = np.mean([nilai(ASLI[s], m)[0] for s in ['sel3', 'sel4']])
    aa = np.mean([nilai(ASLIA[s], m)[0] for s in ['sel3', 'sel4']])
    arah = A.ARAH_BAIK.get(m, 'max')
    best = min([('PURE3', p), ('ASLI', a), ('ASLI+acc', aa)],
               key=lambda x: x[1] if arah == 'min' else -x[1])[0]
    print(f'  {m:14s} PURE3={p:9.4f}  ASLI={a:9.4f}  ASLI+acc={aa:9.4f}  -> {best}')
print()
print('CATATAN untuk Bab V:')
print('  - Keunggulan PURE3 pada pemerataan harus dibaca bersama ongkos koordinasinya.')
print('  - Klaim "alpha lenyap" bersifat STRUKTURAL (bukti aljabar), didukung tak')
print('    langsung oleh kestabilan bobot DGR -- bukan diuji langsung.')

=============================RINGKASAN BLOK D=============================

Basis perbandingan: sel3 & sel4 (satu-satunya yang ada di ketiga varian).
Faktorial ASLI TIDAK lengkap -> efek utama & interaksi tak dapat dihitung.

  gini_UTIL      PURE3=   0.0364  ASLI=   0.0271  ASLI+acc=   0.0246  -> ASLI+acc


  wait           PURE3=  75.1782  ASLI=  70.9674  ASLI+acc=  67.7458  -> ASLI+acc
  acc            PURE3=   0.7114  ASLI=   0.7093  ASLI+acc=   0.7170  -> ASLI+acc
  rec_entropy    PURE3=   0.0965  ASLI=   0.1262  ASLI+acc=   0.1284  -> ASLI+acc


  herding        PURE3=   0.3281  ASLI=   0.2914  ASLI+acc=   0.2903  -> ASLI+acc

CATATAN untuk Bab V:
  - Keunggulan PURE3 pada pemerataan harus dibaca bersama ongkos koordinasinya.
  - Klaim "alpha lenyap" bersifat STRUKTURAL (bukti aljabar), didukung tak
    langsung oleh kestabilan bobot DGR -- bukan diuji langsung.
